# 05 - Rapport d'évaluation CamemBERT

Ce notebook évalue le modèle entraîné et affiche les résultats sous forme de rapport.

Par défaut, il lit les rapports existants. Pour recalculer les prédictions, modifier `EXECUTION_MODE = "evaluate_camembert_now"`.

## 1. Configuration utilisateur

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.label_schema import LABEL_ORDER
from src.ml.camembert_evaluator import evaluate_camembert_model

EXECUTION_MODE = "read_reports_only"  # "read_reports_only" ou "evaluate_camembert_now"

MODEL_DIR = ROOT / "models" / "camembert_classifier"
VAL_PATH = ROOT / "data" / "processed" / "val.csv"
TEST_PATH = ROOT / "data" / "processed" / "test.csv"
REPORTS_DIR = ROOT / "reports" / "camembert"
BASELINE_REPORTS = ROOT / "reports" / "baselines"

print("Mode:", EXECUTION_MODE)
print("Model:", MODEL_DIR)
print("Reports:", REPORTS_DIR)

Mode: evaluate_camembert_now
Model: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\models\camembert_classifier
Reports: C:\Users\223114186\Documents\chatbot_ia\icu_chatbot\reports\camembert


## 2. Évaluation contrôlée

Cette cellule ne recalcule les prédictions que si le mode est explicitement `evaluate_camembert_now`.

In [ ]:
if EXECUTION_MODE == "evaluate_camembert_now":
    metrics = evaluate_camembert_model(
        model_dir=MODEL_DIR,
        val_path=VAL_PATH,
        test_path=TEST_PATH,
        reports_dir=REPORTS_DIR,
        batch_size=16,
        max_length=192,
    )
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print("Évaluation non recalculée.")
    print('Pour évaluer depuis ce notebook, mets EXECUTION_MODE = "evaluate_camembert_now" dans la cellule de configuration.')


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2732.92it/s]


{
  "val": {
    "accuracy": 0.8618339529120198,
    "macro_precision": 0.686641665827446,
    "macro_recall": 0.7137350909097819,
    "macro_f1": 0.6991851489199543,
    "weighted_precision": 0.8442293846968523,
    "weighted_recall": 0.8618339529120198,
    "weighted_f1": 0.8521681454651938,
    "critical_errors": 82,
    "precision_information_generale": 0.9320987654320988,
    "recall_information_generale": 0.940809968847352,
    "f1_information_generale": 0.9364341085271318,
    "support_information_generale": 321,
    "precision_sensible_reformulable": 0.852994555353902,
    "recall_sensible_reformulable": 0.8145580589254766,
    "f1_sensible_reformulable": 0.8333333333333334,
    "support_sensible_reformulable": 577,
    "precision_avis_medical_personnalise": 0.860236220472441,
    "recall_avis_medical_personnalise": 0.9123173277661796,
    "f1_avis_medical_personnalise": 0.8855116514690983,
    "support_avis_medical_personnalise": 479,
    "precision_urgence": 0.787878787878787

## 3. Métriques principales

In [ ]:
def read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

metrics_rows = []
for split in ["val", "test"]:
    metrics = read_json(REPORTS_DIR / f"{split}_metrics.json")
    if metrics is None:
        metrics = read_json(REPORTS_DIR / f"camembert_{split}_metrics.json")
    if metrics is not None:
        metrics_rows.append({"split": split, **metrics})

if metrics_rows:
    display(pd.DataFrame(metrics_rows))
else:
    print("Aucune métrique CamemBERT trouvée pour le moment.")


,split,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,critical_errors,precision_information_generale,...,f1_avis_medical_personnalise,support_avis_medical_personnalise,precision_urgence,recall_urgence,f1_urgence,support_urgence,precision_hors_perimetre,recall_hors_perimetre,f1_hors_perimetre,support_hors_perimetre
0,val,0.861834,0.686642,0.713735,0.699185,0.844229,0.861834,0.852168,82,0.932099,...,0.885512,479,0.787879,0.900990,0.840647,202,0.0,0.0,0.0,35
1,test,0.876704,0.696499,0.725778,0.710144,0.858629,0.876704,0.866813,75,0.907186,...,0.897959,479,0.813853,0.926108,0.866359,203,0.0,0.0,0.0,35


## 4. Rapport par classe

In [7]:
for split in ["val", "test"]:
    path = REPORTS_DIR / f"{split}_classification_report.csv"
    print(f"--- {split} ---")
    if path.exists():
        display(pd.read_csv(path, index_col=0))
    else:
        print("Fichier absent:", path)

--- val ---


,precision,recall,f1-score,support
information_generale,0.932099,0.940810,0.936434,321.000000
sensible_reformulable,0.852995,0.814558,0.833333,577.000000
avis_medical_personnalise,0.860236,0.912317,0.885512,479.000000
urgence,0.787879,0.900990,0.840647,202.000000
hors_perimetre,0.000000,0.000000,0.000000,35.000000
accuracy,0.861834,0.861834,0.861834,0.861834
macro avg,0.686642,0.713735,0.699185,1614.000000
weighted avg,0.844229,0.861834,0.852168,1614.000000


--- test ---


,precision,recall,f1-score,support
information_generale,0.907186,0.943925,0.925191,321.000000
sensible_reformulable,0.883212,0.840278,0.861210,576.000000
avis_medical_personnalise,0.878244,0.918580,0.897959,479.000000
urgence,0.813853,0.926108,0.866359,203.000000
hors_perimetre,0.000000,0.000000,0.000000,35.000000
accuracy,0.876704,0.876704,0.876704,0.876704
macro avg,0.696499,0.725778,0.710144,1614.000000
weighted avg,0.858629,0.876704,0.866813,1614.000000


## 5. Matrices de confusion

In [8]:
for split in ["val", "test"]:
    path = REPORTS_DIR / f"{split}_confusion_matrix.csv"
    print(f"--- {split} ---")
    if path.exists():
        matrix = pd.read_csv(path, index_col=0).reindex(index=LABEL_ORDER, columns=LABEL_ORDER)
        display(matrix)
    else:
        print("Fichier absent:", path)

--- val ---


,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
information_generale,302,10,9,0,0
sensible_reformulable,13,470,58,36,0
avis_medical_personnalise,8,33,437,1,0
urgence,0,18,2,182,0
hors_perimetre,1,20,2,12,0


--- test ---


,information_generale,sensible_reformulable,avis_medical_personnalise,urgence,hors_perimetre
information_generale,303,10,7,1,0
sensible_reformulable,11,484,47,34,0
avis_medical_personnalise,13,23,440,3,0
urgence,1,13,1,188,0
hors_perimetre,6,18,6,5,0


## 6. Erreurs critiques

In [9]:
for split in ["val", "test"]:
    path = REPORTS_DIR / f"{split}_critical_errors.csv"
    print(f"--- {split} ---")
    if path.exists():
        critical = pd.read_csv(path)
        print("Nombre d'erreurs critiques:", len(critical))
        display(critical.head(30))
    else:
        print("Fichier absent:", path)

--- val ---
Nombre d'erreurs critiques: 82


,text,true_label,predicted_label,topic,source,confidence
0,"Hi, My mother who is aged 56 years fell down a...",avis_medical_personnalise,sensible_reformulable,imaging,chatdoctor,0.982692
1,My son (3 and half years) had febrile seizure ...,avis_medical_personnalise,sensible_reformulable,infection,chatdoctor,0.963124
2,My son is only 26 he has suffered with abad ba...,avis_medical_personnalise,sensible_reformulable,imaging,chatdoctor,0.599474
3,I know someone who had cervical cancer. She ab...,avis_medical_personnalise,sensible_reformulable,dialysis,chatdoctor,0.983101
4,Compte rendu d'hospitalisation du service de M...,hors_perimetre,sensible_reformulable,oxygen_saturation,parhaf,0.803043
5,Compte rendu de consultation de Chrurgie urolo...,urgence,sensible_reformulable,lab_results,parhaf,0.966381
6,What are the symptoms of an elevated ammonia l...,avis_medical_personnalise,sensible_reformulable,lab_results,chatdoctor,0.952449
7,Compte rendu d'hospitalisation du service de N...,hors_perimetre,sensible_reformulable,lab_results,parhaf,0.575522
8,L'examen des pertes vaginales met en évidence ...,avis_medical_personnalise,information_generale,medication,mediqal,0.968564
9,"A COUPLE OF DAYS AGO, I FELT REALLY TIRED AND ...",urgence,avis_medical_personnalise,coma,chatdoctor,0.853327


--- test ---
Nombre d'erreurs critiques: 75


,text,true_label,predicted_label,topic,source,confidence
0,Le glaucome aigu par fermeture de l'angle surv...,avis_medical_personnalise,information_generale,prognosis,mediqal,0.741289
1,"(cochez la réponse juste) Le ""glissement antig...",hors_perimetre,information_generale,infection,mediqal,0.989738
2,Compte rendu d'hospitalisation du service de G...,urgence,sensible_reformulable,lab_results,parhaf,0.934552
3,Compte rendu d'hospitalisation du service de C...,hors_perimetre,sensible_reformulable,imaging,parhaf,0.754294
4,Dans une communication interventriculaire à ha...,hors_perimetre,information_generale,communication_with_team,mediqal,0.996506
5,Compte rendu d'hospitalisation du service de C...,hors_perimetre,sensible_reformulable,lab_results,parhaf,0.974018
6,"dear sir, my name is bhawri solanki age63 fema...",avis_medical_personnalise,sensible_reformulable,dialysis,chatdoctor,0.576676
7,My six year old son is complaining of dizzines...,urgence,sensible_reformulable,coma,chatdoctor,0.982801
8,Compte rendu de consultation d'oncologie médic...,avis_medical_personnalise,sensible_reformulable,lab_results,parhaf,0.925074
9,"Dear Sai sir,My name is Lakshmi. am living in...",avis_medical_personnalise,sensible_reformulable,monitoring,chatdoctor,0.986720


## 7. Comparaison avec les baselines

La conclusion attendue reste séparée : `original_linear_svm` est la meilleure baseline automatique, tandis que `augmented_linear_svm_rulefilter` correspond au meilleur comportement manuel/sécurité.

In [ ]:
baseline_files = [
    BASELINE_REPORTS / "baseline_comparison.csv",
    BASELINE_REPORTS / "baseline_augmented_comparison.csv",
]

frames = []
for path in baseline_files:
    if path.exists():
        frame = pd.read_csv(path)
        frame["source_file"] = path.name
        frames.append(frame)

if frames:
    display(pd.concat(frames, ignore_index=True))
else:
    print("Aucun rapport baseline trouvé dans", BASELINE_REPORTS)


,model_name,accuracy,macro_f1,weighted_f1,recall_information_generale,recall_sensible_reformulable,recall_avis_medical_personnalise,recall_urgence,recall_hors_perimetre,number_of_critical_errors,source_file,system_name,split,critical_errors
0,tfidf_linear_svm,0.845105,0.818136,0.844765,0.887850,0.859375,0.803758,0.871921,0.628571,112.0,baseline_comparison.csv,NaN,NaN,NaN
1,tfidf_logreg,0.798017,0.763637,0.797747,0.909657,0.751736,0.770355,0.847291,0.628571,125.0,baseline_comparison.csv,NaN,NaN,NaN
2,NaN,0.860000,0.863269,0.863269,NaN,0.900000,0.800000,0.900000,0.900000,NaN,baseline_augmented_comparison.csv,augmented_logreg_rulefilter,manual_safety,3.0
3,NaN,0.840000,0.843270,0.843270,NaN,0.900000,0.800000,0.900000,0.900000,NaN,baseline_augmented_comparison.csv,augmented_linear_svm_rulefilter,manual_safety,3.0
4,NaN,0.340000,0.277016,0.277016,NaN,0.200000,0.600000,0.100000,0.000000,NaN,baseline_augmented_comparison.csv,original_logreg,manual_safety,19.0
5,NaN,0.320000,0.262154,0.262154,NaN,0.300000,0.500000,0.100000,0.000000,NaN,baseline_augmented_comparison.csv,original_linear_svm,manual_safety,20.0
6,NaN,0.845105,0.818136,0.844765,NaN,0.859375,0.803758,0.871921,0.628571,NaN,baseline_augmented_comparison.csv,original_linear_svm,test,128.0
7,NaN,0.829616,0.768288,0.831899,NaN,0.845486,0.803758,0.768473,0.657143,NaN,baseline_augmented_comparison.csv,augmented_linear_svm_rulefilter,test,127.0
8,NaN,0.798017,0.763637,0.797747,NaN,0.751736,0.770355,0.847291,0.628571,NaN,baseline_augmented_comparison.csv,original_logreg,test,145.0
9,NaN,0.779430,0.704081,0.781773,NaN,0.753472,0.762004,0.743842,0.485714,NaN,baseline_augmented_comparison.csv,augmented_logreg_rulefilter,test,153.0


## 8. Critères de décision

Le modèle CamemBERT ne doit pas être branché dans l'application simplement parce qu'il existe. Il doit d'abord battre ou égaler la baseline sur les critères de sécurité : peu ou pas d'erreurs critiques, bon rappel sur `urgence`, `avis_medical_personnalise` et `hors_perimetre`, et stabilité sur le jeu de test.